<div style="display:block;width:100%;margin:auto;" direction=rtl align=center><br><br>
    <div  style="width:100%;margin:100;display:block;background-color:#fff0;"  display=block align=center>
        <table style="border-style:hidden;border-collapse:collapse;">
            <tr>
                <td  style="border: none!important;">
        <img width=230 align=right src="https://i.ibb.co/yXKQmtZ/logo1.png" style="margin:0;" />
                </td>
                <td style="text-align:center;border: none!important;">
                <h1 align=center><font size=5 color="#045F5F"> <b>Alireza Karimi</b><br><br>810101492</i></font></h1>
                </td>
                <td style="border: none!important;">
        <img width=300 align=left  src="https://i.ibb.co/wLjqFkw/logo2.png" style="margin:0;" />
                </td>
            </tr>
        </table>



# Q2

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Word2Vec, Tokenizer, RegexTokenizer
from pyspark.sql.functions import col

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Question2_Word2Vec") \
    .getOrCreate()

In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/dmls/ca3/q2/wiki_corpus'

data_df = spark.read.text(file_path)

data_df.show(10, truncate=False)

+-------------------------------------------------------------------------------------------------+
|value                                                                                            |
+-------------------------------------------------------------------------------------------------+
|was reported before the land was made inaccessible by alabama power caves in the united states   |
|genoese may refer to a person from genoa the genoese dialect raven oak born december two         |
|one one nine seven seven in los angeles california is an american singer pianist and songwriter  |
|whose works deal with the survival side of life as a survivor of both abuse and                  |
|an attempted rape raven oak strove to free herself from the confines of the silence often        |
|present in victims of abuse and rape her first cd walls boxes was written when her               |
|attempts at writing music for college music classes resulted in being shoved into one category or|


truncate=False: This prevents Spark from cutting off long sentences, allowing you to read the full line.

In [4]:
print(f"Total lines: {data_df.count()}")

Total lines: 562826


## 1

In [5]:
# Tokenize the text (Convert sentences to lists of words)
# RegexTokenizer is often better to remove punctuation and split by whitespace
tokenizer = RegexTokenizer(inputCol="value", outputCol="words", pattern="\\W")
tokenized_df = tokenizer.transform(data_df)
tokenized_df.show(10, truncate=False)

+-------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------+
|value                                                                                            |words                                                                                                             |
+-------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------+
|was reported before the land was made inaccessible by alabama power caves in the united states   |[was, reported, before, the, land, was, made, inaccessible, by, alabama, power, caves, in, the, united, states]   |
|genoese may refer to a person from genoa the genoese dialect raven oak born december two         |[genoese, may, refer, to, a, person, from

Don't use simple space splitting (split(..., ' ')). This leaves punctuation attached to words (e.g., "world." and "world" are treated as different words).

I Used RegexTokenizer with pattern \\W. This removes punctuation automatically, resulting in a cleaner vocabulary and better model training.

In [6]:
# Define the Word2Vec Model
word2Vec = Word2Vec(
    vectorSize=100,  # Dimension of the vector (e.g., 100, 200)
    minCount=5,      # Ignore words that appear fewer than 5 times
    windowSize=5,    # Context window size
    maxIter=3,       # Default was 1, and the more it is, the better results we can get
    inputCol="words",
    outputCol="result"
)

# Train the model (This triggers the computation)
print("Training model... this may take time depending on data size.")
model = word2Vec.fit(tokenized_df)

print("Training complete!")

Training model... this may take time depending on data size.
Training complete!


1. vectorSize=100:

What it does: It tells the model to represent every word (like "King" or "Apple") as a list of 100 numbers.

Why it matters: A size of 100 is a standard balance. If it's too small (e.g., 10), the model can't capture complex meanings. If it's too large (e.g., 500) and you have small data, the model effectively "memorizes" instead of learns, and the files get huge.

2. minCount=5:

What it does: It ignores any word that appears fewer than 5 times in the entire text file.

Why it matters: It removes typos and very rare words. This is crucial because you cannot learn a meaningful vector for a word that only appears once or twice; it just adds noise to the model.

3. windowSize=5:

What it does: When the model looks at a word (e.g., "fox" in "The quick brown fox jumps over"), it looks at 5 words to the left and 5 words to the right to understand the context.

Why it matters: This teaches the model that "fox" is related to "quick", "brown", and "jumps". A window of 5 is standard for capturing semantic relationships.

4. maxIter=5 (The Critical Fix):

What it does: It forces the model to read through your entire dataset 5 times. (The default in PySpark is often just 1).

Why it matters: This is the main reason your previous result for "Queen" was #29 while it can be #2 after 20 epochs. One pass (1 epoch) is just "skimming" the text. 20 passes (20 epochs) allows the neural network to adjust the numbers repeatedly until the mathematical relationship (King - Man + Woman = Queen) solidifies.

5. inputCol="words":

This tells Spark to look for the column named "words" in your DataFrame (which you created earlier using the Tokenizer) to find the text data.

6. outputCol="result":

This names the new column where Spark will store the final numerical vectors after processing.

I got help from [this chat](https://gemini.google.com/share/b025a721dcf8) to complete the codes of this part.

It actually takes a lot of time to train, so I decreased the maxIter from 20 to 3 because everytime I got runtime disconnected error.

---

## 2

I actually got help from [this chat](https://gemini.google.com/share/2b9cad867cfe) to complete the codes and know about the neccessary functions needed for this part.

In [7]:
# Get the DataFrame of all vectors
# This DataFrame has two columns: "word" (string) and "vector" (Vector)
vectors_df = model.getVectors()
vectors_df

DataFrame[word: string, vector: vector]

In [8]:
target_words = ["iran", "tehran", "learning", "science"]

In [10]:
result_df = vectors_df.filter(col("word").isin(target_words))
result_df

DataFrame[word: string, vector: vector]

In [11]:
result_df.show(truncate=False)

+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
# (Optional) Convert to a list to copy-paste into report
# This collects the data to the driver and prints it nicely
rows = result_df.collect()

rows = result_df.collect()

What it does: This triggers the computation and moves data from the cluster to your local computer.

Detailed Breakdown:

result_df: This is a Spark DataFrame. It exists distributed across the memory of multiple worker nodes (executors). You cannot print or loop through it directly in standard Python because the data isn't "on" your main computer yet.

.collect(): This is a Spark Action. It tells Spark to gather all the rows from the distributed workers and send them back to the "Driver" (your Python script) as a standard Python list of Row objects.

Why you use it: You use it so you can loop through the data using a standard python for loop to print it nicely.

In [13]:
print(f"\n{'Word':<10} | {'Vector Fragment (First 5 values)'}")
print("-" * 50)

for row in rows:
    word = row['word']
    vector = row['vector']
    # Show only first 5 numbers for brevity in the print output
    vector_preview = str(vector.toArray()[:5])
    print(f"{word:<10} | {vector_preview} ...")


Word       | Vector Fragment (First 5 values)
--------------------------------------------------
iran       | [-0.00412064 -0.00625178 -0.416381    0.5127936  -0.36546201] ...
science    | [ 0.03819476  0.29435104 -0.14244814  0.30367813  0.50213367] ...
tehran     | [ 0.11868411  0.05355461  0.3344177   0.25999284 -0.16858964] ...
learning   | [-0.02768707  0.01350845 -0.03845796  0.1454476   0.1877203 ] ...


---

## 3

I actually got help from [this chat](https://gemini.google.com/share/cd043bacda92) to complete the codes of this part.

In [14]:
target_words = ["iran", "tehran", "learning", "science"]

print(f"{'Target Word':<15} | {'Synonym':<20} | {'Cosine Similarity':<20}")
print("-" * 60)

for word in target_words:
    try:
        synonyms_df = model.findSynonyms(word, 5)
        synonyms_results = synonyms_df.collect()

        for row in synonyms_results:
            print(f"{word:<15} | {row['word']:<20} | {row['similarity']:<20.4f}")
        print("-" * 60)

    except Exception as e:
        print(f"Could not find synonyms for '{word}'. Reason: {str(e)}")
        print("-" * 60)

Target Word     | Synonym              | Cosine Similarity   
------------------------------------------------------------
iran            | pakistan             | 0.7751              
iran            | afghanistan          | 0.7607              
iran            | egypt                | 0.6933              
iran            | ethiopia             | 0.6833              
iran            | tunisia              | 0.6769              
------------------------------------------------------------
tehran          | adana                | 0.7087              
tehran          | sheremetyevo         | 0.6974              
tehran          | ltu                  | 0.6874              
tehran          | luxair               | 0.6808              
tehran          | istanbul             | 0.6800              
------------------------------------------------------------
learning        | numeracy             | 0.6507              
learning        | educational          | 0.6494              
learning   

synonyms_df = model.findSynonyms(word, 5)

What it does: It asks the model to find the "nearest neighbors" to a specific word in the mathematical vector space.

Detailed Breakdown:

model: The trained Word2Vec model that learned where every word lives in the vector space.

word: The query word (e.g., "iran" or "tehran").

5: The number of synonyms to return.

How it works: It calculates the Cosine Similarity between the vector of your input word and every other word in the vocabulary. It returns a DataFrame containing the top 5 matches and their similarity scores.

---

## 4

In [15]:
import numpy as np
from pyspark.sql.functions import col

In [16]:
words_to_extract = ["king", "man", "woman", "queen"]

# Extract vectors for these specific words
# Collect them into a dictionary for easy access: {'king': [vector], 'man': [vector], ...}
vectors_df = model.getVectors().filter(col("word").isin(words_to_extract))
word_vec_map = {row['word']: np.array(row['vector'].toArray()) for row in vectors_df.collect()}

1. vectors_df = model.getVectors().filter(col("word").isin(words_to_extract))

What it does: It retrieves the numerical vectors only for a specific list of words you care about (e.g., King, Man, Woman, Queen).

Detailed Breakdown:

model.getVectors(): Returns a massive DataFrame containing every word in the vocabulary and its vector representation.

col("word").isin(words_to_extract): A condition that checks if the value in the "word" column exists in your list ["king", "man", "woman", "queen"].

.filter(...): Keeps only the rows that satisfy that condition, discarding the thousands of other words you don't need right now.

2. word_vec_map = {row['word']: np.array(row['vector'].toArray()) for row in vectors_df.collect()}

What it does: It converts the Spark data into a Python dictionary of NumPy arrays so you can do math with them (addition/subtraction).

Detailed Breakdown:

vectors_df.collect(): Brings the filtered rows (King, Man, Woman, Queen) to your local Python environment.

row['word']: Gets the word string (e.g., "king") to use as the dictionary key.

row['vector'].toArray(): Converts the Spark Vector object into a standard Python list.

np.array(...): Converts that list into a NumPy array. This is crucial because Python lists don't support vector math (you can't do listA - listB), but NumPy arrays do.

Result: You get a dictionary like {'king': array([0.1, 0.5...]), 'man': array([...])}, which allows you to run the formula: target = word_vec_map["king"] - word_vec_map["man"] + word_vec_map["woman"].

In [17]:
# e(king) - e(man) + e(woman)
target_vector = word_vec_map["king"] - word_vec_map["man"] + word_vec_map["woman"]

The reason the formula `vector("king") - vector("man") + vector("woman")` works is because of how Word2Vec learns to map meaning to **geometry (direction and distance)**.

Here is the step-by-step breakdown of why this simple math equals "Queen":

### 1. Meaning is broken into "Components"

Imagine that every word is just a bundle of concepts. For example:

* **King**  [ Royalty ] + [ Male ]
* **Queen**  [ Royalty ] + [ Female ]
* **Man**  [ Male ]
* **Woman**  [ Female ]

The Word2Vec model doesn't label them "Royalty" or "Male" explicitly, but it learns **directions** in the vector space that correspond to these concepts.

### 2. The Math Operation

If we substitute those concepts into your formula, look at what happens:

1. **Step 1:** `King - Man`
*
* You have effectively removed the "maleness" from King, leaving just the pure concept of "Ruler/Monarch".


2. **Step 2:** `+ Woman`
*
* You take that pure "Ruler" concept and add "femaleness" to it.


3. **Result:**
* The word that best represents  is **Queen**.



### 3. The "Gender Arrow"

Visually, the model learns a consistent **"Gender Vector"**.
If you draw a line from **Man** to **Woman**, that line points in a specific direction (let's call it "South-East").
The model learns that *any* male word can be turned into a female word by moving "South-East" by that same distance.

* `Uncle` + (Gender Vector)  `Aunt`
* `Brother` + (Gender Vector)  `Sister`
* `King` + (Gender Vector)  `Queen`

Your formula is just a rearrangement of this idea:


### Why it failed with 1 Epoch (Rank 29) vs. 20 Epochs (Rank 2)

For this math to work, the "Gender Arrow" for *King  Queen* must be **parallel** to the arrow for *Man  Woman*.

* **1 Epoch:** The model vaguely knows King and Queen are related, and Man and Woman are related, but the arrows point in slightly different directions. The math doesn't line up, so you land in a random spot (Rank 29).
* **20 Epochs:** The model forces all these pairs to align perfectly parallel to minimize error. The math lines up, and you land right on "Queen" (Rank 2).

In [18]:
# We ask for top 30 to see where 'queen' lands
target_synonyms = model.findSynonyms(target_vector, 30).collect()

In [19]:
print(f"Finding neighbors for vector (King - Man + Woman)...")
print(f"{'Rank':<5} | {'Word':<15} | {'Similarity':<20}")
print("-" * 45)
for i, row in enumerate(target_synonyms):
    print(f"{i+1:<5} | {row['word']:<15} | {row['similarity']:<20.4f}")

Finding neighbors for vector (King - Man + Woman)...
Rank  | Word            | Similarity          
---------------------------------------------
1     | king            | 0.8018              
2     | prince          | 0.6827              
3     | isabella        | 0.6683              
4     | consort         | 0.6567              
5     | queen           | 0.6527              
6     | birendra        | 0.6424              
7     | farouk          | 0.6255              
8     | coronation      | 0.6235              
9     | philip          | 0.6196              
10    | wenceslas       | 0.6069              
11    | elizabeth       | 0.5932              
12    | niece           | 0.5925              
13    | bikram          | 0.5924              
14    | arnulf          | 0.5875              
15    | aishwarya       | 0.5862              
16    | widowed         | 0.5766              
17    | monarch         | 0.5743              
18    | gyanendra       | 0.5732              
19    | 

As you can see, "queen" is the number 29 for 1 epoch and it is 5 in 3 epoch and it is 2 in 20 epock I believe, in the list of similar words based on the target_vector.

In [20]:
# Calculate Cosine Similarity manually between target_vector and queen_vector
queen_vec = word_vec_map["queen"]

In [21]:
# Cosine Similarity Formula: (A . B) / (||A|| * ||B||)
dot_product = np.dot(target_vector, queen_vec)
norm_target = np.linalg.norm(target_vector)
norm_queen = np.linalg.norm(queen_vec)

cosine_sim = dot_product / (norm_target * norm_queen)

print("-" * 45)
print(f"Direct Similarity to 'queen': {cosine_sim:.4f}")

---------------------------------------------
Direct Similarity to 'queen': 0.6527


I got some of the explanation of the codes in question 2 from [this chat](https://gemini.google.com/share/2948b0d1516b)

Based on the Word2Vec algorithm and the formula , here is the explanation for why increasing `maxIter` (training epochs) dramatically improves the ranking of "Queen".

### 1. The Starting Point: Random Chaos

At the very beginning (before training), every word vector is initialized with **random numbers**.

* All words are scattered randomly in the vector space.
* The math  results in a random vector that points nowhere meaningful.

### 2. After 1 Epoch: "Rough Draft" (Rank 29)

When you train for just **1 Epoch**, the model reads through the Wikipedia corpus exactly **once**.

* **What it learns:** It learns basic associations (e.g., "King" appears near "palace" or "throne"). It pulls related words closer together.
* **The Problem:** It hasn't seen the data enough times to fine-tune the *angles* and *distances* between vectors.
* **The Result:** "King", "Man", "Woman", and "Queen" are somewhat loosely grouped, but the specific **gender direction** () is messy.
* Subtracting "Man" and adding "Woman" shifts the vector, but because the "gender direction" isn't perfectly aligned yet, the result lands in a "noisy" neighborhood. It might land closer to generic royal words (like "Prince" or names of specific kings) than to "Queen". Hence, Rank 29.



### 3. Increasing Epochs (e.g., 3 to 20): "Fine-Tuning"

Word2Vec uses an optimization technique called **Stochastic Gradient Descent (SGD)**. In each epoch, it creates a "penalty" every time it predicts a word incorrectly and nudges the vectors slightly to fix the error.

* **Aligning the Subspaces:** With more iterations (20 epochs), the model repeatedly sees pairs like *King/Queen*, *Man/Woman*, *Brother/Sister*, *Uncle/Aunt*.
* **The "Gender" Arrow:** The model realizes that to minimize error across *all* these pairs, it must arrange them so the distance/direction between *Male* and *Female* versions is **consistent**.
* The vector for () becomes almost parallel to ().


* **The Result:** Now, when you calculate :
1. You start at **King**.
2. You move "backwards" along the Male vector (removing the "Man" concept).
3. You move "forwards" along the Female vector (adding the "Woman" concept).
4. Because the geometry is now highly organized, this path leads you precisely to the coordinate for **Queen**.



### Summary Visualization

Imagine trying to draw a perfect square on a piece of paper:

* **1 Epoch:** You draw it quickly without lifting your pen. The lines are wobbly. If you try to navigate corners, you miss the start point. (Result: Rank 29).
* **20 Epochs:** You trace over the lines 20 times, correcting the wiggles each time. The shape becomes a sharp, perfect square. Navigating the corners leads you exactly where you expect. (Result: Rank 2).

---